# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path
from pydantic import BaseModel, Field

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs
print('DATA_DIR is:',DATA_DIR)

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])
print('golden set loaded is:',golden)

DATA_DIR is: ../data
Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
golden set loaded is: {'j01': {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'clean — clear company + role + years'}, 'j02': {'id': 'j02', 'company': 'Northwind Ltd', 'role': 'Data Analyst', 'years_experience_required': 2, 'notes': 'clean — preferred used but still a stated minimum'}, 'j03': {'id': 'j03', 'company': 'Globex International', 'role': 'Product Manager - Growth', 'years_experience_required': 4, 'notes': "clean — 'at least 4 years'"}, 'j04': {'id': 'j04', 'company': 'Initech', 'role': 'Lead DevOps Engineer', 'years_experience_required': 6, 'notes': "slightly fuzzy — 'around 6 years' — accept 6"}, 'j05': {'id': 'j05'

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the company name, job role, and number of years of experience
required from the following job description.

Return the response only in JSON with these fields:
company, role, years_experience_required.

Job description:{snippet_text}"""
        }
    ]







def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include worked examples in the prompt."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the hiring company name, job role, and number of years
of experience required from the job description.

Return the response ONLY in JSON with these fields:
company, role, years_experience_required.

Example:

Job description:
ABC is hiring an ML Engineer with 5 years of experience
in machine learning and Python.

Output:
{{
    "company": "ABC",
    "role": "ML Engineer",
    "years_experience_required": 5
}}

If no years of experience are mentioned, return null.

Now extract the information from the following job description:

{snippet_text}

Return ONLY the JSON response.
"""
        }
    ]







def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based.
    Use a system prompt with a persona and explicit JSON schema.
    """

    return [
        {
            "role": "system",
            "content": """
ROLE:
You are a job extraction expert and recruitment assistant.

TASK:
Extract the hiring company name, job role, and number of years
of experience required from the job description.

If the required information is not present in the context,
return null for the missing value.
Do not invent or hallucinate information.

EXAMPLE:

Job description:
ABC is hiring an ML Engineer with 5 years of experience
in machine learning and Python.

Output:
{
    "company": "ABC",
    "role": "ML Engineer",
    "years_experience_required": 5
}

FORMAT:
Return ONLY valid JSON using exactly these fields:

{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null"
}
"""
        },
        {
            "role": "user",
            "content": f"""
Job description:

{snippet_text}
"""
        }
    ]








def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            "role": "system",
            "content": """
You are a careful job extraction expert and recruitment assistant.

TASK:
Analyze the job description step by step and identify:

1. The hiring company name
2. The job role
3. The required skillset
4. The minimum number of years of experience required

REASONING RULES:
- Carefully inspect the full job description before answering.
- Identify the company name only if it is explicitly stated.
- Identify the exact job role.
- Extract only the skillsets mentioned in the job description.
- Determine the minimum years of experience required.
- For "5+ years", return 5.
- For "at least 4 years", return 4.
- For "around 6 years", return 6.
- For "about three years", return 3.
- For a range such as "3-5 years", return 3.
- If fresh graduates are accepted, return 0.
- If years of experience are not mentioned, return null.
- If any other information is missing, return null.
- Do not guess or hallucinate information.

Think through the extraction carefully before producing the answer,
but return ONLY the final JSON result.

FORMAT:
Return only valid JSON with exactly these fields:

{
    "company": "string or null",
    "role": "string or null",
    "skillset": "string or null",
    "years_experience_required": "integer or null"
}
"""
        },
        {
            "role": "user",
            "content": f"""
Analyze the following job description carefully:

{snippet_text}

Return only the final JSON response.
"""
        }
    ]




STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
import random

def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    # TODO: extract + parse the JSON, return a dict or None
    
    print("Executing parse_response ::",text)

    text = text.strip()

    if text.startswith("```json"):
        text = text[7:]

    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    parsed = json.loads(text)
    print("Response parsed is::", parsed)

    if parsed is not None:
        print("Since parsed response is not null mapping the fields")
        company = parsed.get("company")
        role = parsed.get("role")
        years_experience_required = parsed.get(
            "years_experience_required"
        )

        print("Company:", company)
        print("Role:", role)
        print("Years:", years_experience_required)


    else:
        print("Parsed response is null")
        company = None
        role = None
        years_experience_required = None

    print("final parsed response is::",company,role,years_experience_required)
    return parsed


async def run_one(strategy_name: str, snippets: list[dict]) -> list[dict]:
    """Run one strategy against all job snippets asynchronously."""

    #Iterating one strategy (zero shot) against each snippets and use gather for batch processing
    activity = [
        run_each_snippet(strategy_name, snippet)
        for snippet in snippets
    ]

    results = await asyncio.gather(*activity)

    return results




async def run_all(snippets: list[dict]) -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # TODO: build the task list, gather, return results
    activity = []

    #Iterating each strategy against each snippets and use gather for batch processing
    for strategy_name in STRATEGIES:
        for snippet in snippets:
            activity.append(run_each_snippet(strategy_name,snippet))

    results = await asyncio.gather(
        *activity,
        return_exceptions=True
    )

    return results





# from types import SimpleNamespace
async def run_each_snippet(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    
    print("strategy name is:::", strategy_name)
    print("snippet id is::::", snippet["id"])
    print("snippet is::::", snippet["snippet"])

    prompt_func_name = STRATEGIES[strategy_name]
    prompt = prompt_func_name(snippet["snippet"])
    print("prompt fetched is::::", prompt)

    #start calculating latency
    start_time = time.perf_counter()
    #await asyncio.sleep(random.uniform(0.3, 1.5))


    resp = await client.chat.completions.create(
        model=MODEL,
       messages=prompt,
        temperature=0.0
    )
    elapsed_time = time.perf_counter() - start_time

    print(f"Latency: {elapsed_time:.4f} seconds")
    print("response from LLM is:::", resp)

    raw_response = resp.choices[0].message.content
    usage = resp.usage
    print("response usage from LLM is:::", usage)

    cost = compute_cost_usd(
                MODEL,
                usage.prompt_tokens if usage else 0,
                usage.completion_tokens if usage else 0,
        )

    # Parse JSON returned by LLM
    #parsedresponse = parse_response(raw_response["content"])
    parsedresponse = parse_response(raw_response)
    print("parsedresponse is:::",parsedresponse)
    
    # Return captured result
    return {
        "id": snippet["id"],
        "strategy": strategy_name,
        "company": parsedresponse["company"],
        "role": parsedresponse["role"],
        "years_experience_required":  parsedresponse["years_experience_required"],
        "raw_response": raw_response,
        "parsed_response": parsedresponse,
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "total_tokens": usage.total_tokens if usage else 0,
        "cost_usd": cost,
        "latency_seconds": elapsed_time
    }


def compute_cost_usd(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    """Convert a usage tuple into USD.

    Returns 0.0 for unknown models rather than raising — a missing rate
    shouldn't break a batch run; it just means we can't price that call.
    """
    print("prompt_tokens::::",prompt_tokens, "completion_tokens::::",completion_tokens, "model::::",model)

    rates = RATES.get(model)

    if rates is None:
        return 0.0
    in_rate = rates['in']
    out_rate = rates['out']
    final_cost = (prompt_tokens * in_rate + completion_tokens * out_rate)
    print(f"${final_cost:.8f}")
    return final_cost #return (f"${final_cost:.8f}")
    #return (prompt_tokens * in_rate + completion_tokens * out_rate) / 1_000_000.0

In [5]:
# Run it

# results = await run_one("zero_shot", snippets)
# print("final result is:::",results)

results = await run_all(snippets)
print(f'Got {len(results)} results.')
results[0]

# Write results to output file
OUTPUT_FILE = DATA_DIR / "output5.txt"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for result in results:
        f.write(str(result))
        f.write("\n")

print("Results written to:", OUTPUT_FILE)

strategy name is::: zero_shot
snippet id is:::: j01
snippet is:::: Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.
prompt fetched is:::: [{'role': 'user', 'content': '\nExtract the company name, job role, and number of years of experience\nrequired from the following job description.\n\nReturn the response only in JSON with these fields:\ncompany, role, years_experience_required.\n\nJob description:Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}]


strategy name is::: zero_shot
snippet id is:::: j02
snippet is:::: We're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred.
prompt fetched is:::: [{'role': 'user', 'content': "\nExtract the company name, job role, and number of years of experience\nrequired from the following job description.\n\nReturn the response only in JSON with these fields:\ncompany, role, years_experience_required.\n\nJob description:We're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred."}]
strategy name is::: zero_shot
snippet id is:::: j03
snippet is:::: Globex International seeks a Product Manager - Growth. You'll own activation and onboarding metrics across our consumer apps. We're looking for someone with at least 4 years of product management experience

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [6]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    # TODO: count exact matches (with normalisation)
    
    if extracted is None:
        return 0

    score = 0

    # Compare company
    extracted_company = extracted.get("company")
    gold_company = gold.get("company")
    print("extraced companu is:::",extracted_company)
    print("gold_account is:::",gold_company)

    if isinstance(extracted_company, str):
        extracted_company = extracted_company.strip().lower()

    if isinstance(gold_company, str):
        gold_company = gold_company.strip().lower()

    if extracted_company == gold_company:
        score = score + 1

    # Compare role
    extracted_role = extracted.get("role")
    gold_role = gold.get("role")

    if isinstance(extracted_role, str):
        extracted_role = extracted_role.strip().lower()

    if isinstance(gold_role, str):
        gold_role = gold_role.strip().lower()

    if extracted_role == gold_role:
        score = score + 1

    # Compare years
    extracted_years = extracted.get(
        "years_experience_required"
    )

    gold_years = gold.get(
        "years_experience_required"
    )

    if extracted_years == gold_years:
        score = score + 1

    return score

RUBRIC = """Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated
  
FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read"""

print(RUBRIC)

class JudgeVerdict(BaseModel):
    accuracy: int      = Field(ge=1, le=4)
    groundedness: int  = Field(ge=1, le=4)
    format_score: int  = Field(ge=1, le=4)
    reasoning: str     = Field(min_length=20, max_length=500)

print("Schema fields:")
for name, field in JudgeVerdict.model_fields.items():
    print(f"  {name:15s}  {field.annotation.__name__}")





async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict, ideal: str = "") -> JudgeVerdict:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score
    
    if extracted is None:
        return JudgeVerdict(
            accuracy=1,
            groundedness=1,
            format_score=1,
            reasoning="The extracted response is missing or could not be parsed."
        )
    resp = await client.chat.completions.parse(
        model=JUDGE_MODEL,
        temperature=0.0,
        response_format=JudgeVerdict,
        messages=[
            {"role": "system", "content":
                f"You are a strict evaluator of LLM answers.\n\n{RUBRIC}\n\n"
                "Return your scores and a brief reasoning explaining the scores."},
        {
                "role": "user",
                "content": f"""
Original job snippet:
{snippet_text}

Golden answer:
{json.dumps(gold, indent=2)}

Candidate extracted answer:
{json.dumps(extracted, indent=2)}
"""
            },
        ],
    )
    print("Judge response is::",resp)

    # resp = ""
    # raw_response = {
    # "id": "chatcmpl-EEGWRe7DU51LvNoTIfHOAyUTECjy3",

    # "content": """```json
    # {
    #   "company": "Acme Corp",
    #   "role": "Senior Software Engineer",
    #   "years_experience_required": 5
    # }```""",

    # "model": "gpt-4o-mini-2024-07-18",

    # "usage": {
    #     "completion_tokens": 34,
    #     "prompt_tokens": 88,
    #     "total_tokens": 122
    # }
    # }
    return resp.choices[0].message.parsed


Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated
  
FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read
Schema fields:
  accuracy         int
  groundedness     int
  format_score     int
  reasoning        str


In [7]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
# LLM Judge score

print(type(golden))
print(golden)

print(type(snippets))
print(type(snippets[0]))
print(snippets[0])


golden_lookup = golden

snippet_lookup = {
    item["id"]: item["snippet"]
    for item in snippets
}

print("Golden j01:", golden_lookup["j01"])
print("Snippet j01:", snippet_lookup["j01"])


scored = []

for result in results:

    if isinstance(result, Exception):
        print("Skipping failed result:", result)
        continue

    goldenset_record = golden_lookup.get(result["id"])

    if goldenset_record is None:
        print("Golden record not found:", result["id"])
        continue

    extracted = {
        "company": result["company"],
        "role": result["role"],
        "years_experience_required":
            result["years_experience_required"]
    }

    accuracy_score = score_accuracy(
        extracted,
        goldenset_record
    )

    parse_success = (
        1 if result.get("parsed_response") is not None else 0
    )

    snippet_text = snippet_lookup[result["id"]]

    verdict = await score_llm_judge(
        snippet_text,
        extracted,
        goldenset_record
    )

    scored.append({
        "id": result["id"],
        "strategy": result["strategy"],
        "accuracy": accuracy_score,
        "parse_success": parse_success,
        "llm_judge_score": verdict.accuracy,
        "groundedness": verdict.groundedness,
        "format_score": verdict.format_score,
        "judge_reasoning": verdict.reasoning,
        "cost_usd": result["cost_usd"],
        "latency_s": result["latency_seconds"]
    })

print("Total scored records:", len(scored))

# Write results to output file
OUTPUT_FILE = DATA_DIR / "judge_response2.txt"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for result in scored:
        f.write(str(result))
        f.write("\n")

print("Results written to:", OUTPUT_FILE)

<class 'dict'>
{'j01': {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'clean — clear company + role + years'}, 'j02': {'id': 'j02', 'company': 'Northwind Ltd', 'role': 'Data Analyst', 'years_experience_required': 2, 'notes': 'clean — preferred used but still a stated minimum'}, 'j03': {'id': 'j03', 'company': 'Globex International', 'role': 'Product Manager - Growth', 'years_experience_required': 4, 'notes': "clean — 'at least 4 years'"}, 'j04': {'id': 'j04', 'company': 'Initech', 'role': 'Lead DevOps Engineer', 'years_experience_required': 6, 'notes': "slightly fuzzy — 'around 6 years' — accept 6"}, 'j05': {'id': 'j05', 'company': 'Hooli', 'role': 'Junior Frontend Developer', 'years_experience_required': 0, 'notes': 'edge — fresh grads OK = 0 years'}, 'j06': {'id': 'j06', 'company': 'Pied Piper Inc.', 'role': 'Senior ML Engineer', 'years_experience_required': 7, 'notes': "fuzzy — '7+ years' typically taken as 7; excep

Judge response is:: ParsedChatCompletion[~ResponseFormatT](id='chatcmpl-EFhcUEDwzWN3yX3E6SfXsU5xGlYoV', choices=[ParsedChoice[~ResponseFormatT](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[~ResponseFormatT](content='{"accuracy":4,"groundedness":4,"format_score":4,"reasoning":"The candidate\'s extracted answer is fully accurate and matches the information provided in the original job snippet. It correctly identifies the company, role, and years of experience required. The answer is also well-grounded as it directly reflects the content of the corpus without any additions or omissions. The format is clear, concise, and well-structured, making it easy to read and understand. Therefore, it receives the highest scores in all dimensions."}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=JudgeVerdict(accuracy=4, groundedness=4, format_score=4, reasoning="The candidate's extracted answer is fully acc

## Step 5 — Build the comparison table

In [8]:
df = pd.DataFrame(scored)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

Rows: 40
Columns: ['id', 'strategy', 'accuracy', 'parse_success', 'llm_judge_score', 'groundedness', 'format_score', 'judge_reasoning', 'cost_usd', 'latency_s']


,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.8,1.0,4.0,0.001,1.154
few_shot,2.7,1.0,3.9,0.000,1.034
structured,2.7,1.0,3.9,0.001,1.087
zero_shot,2.6,1.0,4.0,0.000,1.110


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```